# 频繁子图挖掘流程演示

本笔记本演示 SPMiner 的最小挖掘流程：解析编码器和解码器参数、初始化运行时、加载训练好的 checkpoint，并在一个小规模数据子集上执行一次模式搜索。

In [ ]:
import sys
from pathlib import Path

repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repository root")
repo_root = str(repo_root)
sys.path.insert(0, repo_root)

import argparse

from src.core.cli import setup_runtime
from src.core import dataset_registry
from src.subgraph_matching.config import parse_encoder
from src.subgraph_mining.config import parse_decoder
from src.subgraph_mining import decoder as decoder_mod
from src.logger import RunLogger

In [ ]:
parser = argparse.ArgumentParser(description="挖掘流程演示")
parse_encoder(parser)
parse_decoder(parser)
args = parser.parse_args("")
args.use_gpu = False
args.dataset = "enzymes"
args.model_path = str(Path(repo_root) / "ckpt/model.pt")
args.out_path = str(Path(repo_root) / "results/notebook-patterns.p")
args.batch_size = 8
args.n_neighborhoods = 20
args.n_trials = 3
args.min_pattern_size = 5
args.max_pattern_size = 6
args.min_neighborhood_size = 8
args.max_neighborhood_size = 12
args.search_strategy = "greedy"
args.sample_method = "tree"
args.use_whole_graphs = False
args.analyze = False
args.seed = 0
Path(args.out_path).parent.mkdir(parents=True, exist_ok=True)
device = setup_runtime(args)
print({"device": str(device), "dataset": args.dataset, "model_path": args.model_path, "out_path": args.out_path})

In [ ]:
normalized_dataset = dataset_registry.validate_dataset_name(args.dataset, "mining")
args.dataset = normalized_dataset
dataset, task = dataset_registry.load_dataset_for_stage(args.dataset, "mining")
dataset = list(dataset)[:6]
print({"dataset_size": len(dataset), "task": task})

In [ ]:
with RunLogger(args):
    decoder_mod.pattern_growth(dataset, task, args)
print({"saved_to": args.out_path})